In [1]:
import pandas as pd
import pyterrier as pt
if not pt.started():
    pt.init()
import pyterrier_pisa
from pyterrier_t5 import *
import os
from ir_measures import *

/tmp/ipykernel_3291037/270995605.py:3: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_3291037/270995605.py:4: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()
/opt/miniconda3/envs/tmp/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ohsumed_docs = pd.read_pickle("/nfs/primary/sas_reranker/ohsumed_docs_w_t5base_sensitivity.pkl")
queries = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/queries.pkl")
qrels = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/qrels.pkl")

def _sens_docs(qrels, run):
    if "sensitivity" in run.columns:
        run = run.drop(columns = ["sensitivity"])
    merged = pd.merge(run, ohsumed_docs, left_on = "doc_id", right_on = "docno")
    return merged.sensitivity.sum()
    
import ir_measures
sens_docs = ir_measures.define_byquery(
    _sens_docs, 
    name="sens_docs")

In [3]:
bm25 = pt.Transformer.from_df(pt.io.read_results("/nfs/primary/standard_run_files/ohsumed/bm25_pisa_1000.run"))

In [38]:
def _ns_recall(qrels, run):
    combined = pd.merge(run, qrels, on = ["doc_id"])
    combined = pd.merge(combined, ohsumed_docs, left_on = "doc_id", right_on = "docno")
    num_rel_ns = combined[(combined['sensitivity'] < 1) & (combined['relevance'] > 1)].shape[0]
    qrels_w_sensitivity = pd.merge(qrels, ohsumed_docs, left_on = "doc_id", right_on = "docno")
    total = qrels_w_sensitivity[(qrels_w_sensitivity['sensitivity'] < 1) & (qrels_w_sensitivity['relevance'] > 1)].shape[0]
    assert num_rel_ns <= total
    if total == 0:
        return 0
    return num_rel_ns / total
    
import ir_measures
NSR = ir_measures.define_byquery(
    _ns_recall, 
    name="NSR")

exp = pt.Experiment(
    [bm25],
    queries,
    qrels,
    eval_metrics = [
        nDCG@10,
        sens_docs@10,
        NSR@1000
    ],
    names = ["BM25"],
    verbose = True,
)

exp

pt.Experiment: 100%|██████████| 1/1 [00:00<00:00,  1.41system/s]


,name,nDCG@10,sens_docs@10,NSR@1000
0,BM25,0.391683,1.339623,0.843981


In [42]:
import os
directory_path = './runs'

# List all files (excluding directories)
files = [f for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f)) and "csv" not in f]

names = []
run_files = []

for file in files:
    names.append(file)
    print(f"./runs/{file}")
    run_files.append(pt.io.read_results(f"./runs/{file}"))

./runs/bm25 % 1000 >> filter % 100 >> get_text >> monot5_relevance
./runs/bm25 % 100 >> get_text >> monot5_sans
./runs/bm25 % 1000 >> filter % 100 >> get_text >> monot5_sans
./runs/splade_relevance % 1000 >> filter % 100 >> get_text >> monot5_relevance
./runs/splade_relevance % 100 >> get_text >> monot5_sans
./runs/splade_relevance % 1000 >> filter % 100 >> get_text >> monot5_sans
./runs/splade_sans % 1000 >> filter % 100 >> get_text >> monot5_relevance
./runs/splade_sans % 100 >> get_text >> monot5_sans
./runs/splade_sans % 1000 >> filter % 100 >> get_text >> monot5_sans
./runs/splade_distil % 1000 >> filter % 100 >> get_text >> monot5_relevance >> kill
./runs/splade_distil % 100 >> get_text >> monot5_sans >> kill
./runs/splade_distil % 1000 >> filter % 100 >> get_text >> monot5_sans >> kill
./runs/bm25 % k
./runs/splade_relevance % k
./runs/splade_sans % k
./runs/splade_distil % 100 >> kill
./runs/bm25 % 500 >> filter % 100 >> get_text >> monot5_relevance
./runs/bm25 % 500 >> filter 

In [110]:
exp.to_csv("./runs/results.csv", mode='a', header=not os.path.exists("./runs/results.csv"), index=False)